In [3]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *

In [11]:
def svg_plot(da,savepath,landmask=False,cmap = "RdBu_r",vmin=np.nan,vcenter=np.nan,vmax=np.nan,cbar=False,norm_map=False):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    # --- 2. Set up the projection ---
    central_lon = 210
    proj = ccrs.Robinson(central_longitude=central_lon)

    fig = plt.figure(figsize=(11, 6))
    ax = plt.axes(projection=proj)

    data = da.data
    if norm_map:
        data = data = data/np.std(data)

    # --- 3. Plot the data ---
    # transform=ccrs.PlateCarree() tells cartopy your data is in regular lat/lon coords
    if not np.isnan(vmin):
        mesh = ax.pcolormesh(
                da.lon, da.lat, data,
                transform=ccrs.PlateCarree(),
                cmap=cmap,
                shading="auto",
                norm=mpl.colors.TwoSlopeNorm(vmin=vmin,vcenter=vcenter,vmax=vmax)
            )
    elif not np.isnan(vcenter):
        mag = np.max(np.abs(data))
        mesh = ax.pcolormesh(
            da.lon, da.lat, data,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            norm=mpl.colors.TwoSlopeNorm(vmin=-mag,vcenter=vcenter,vmax=mag),
            shading="auto",
        )
    else:
        mesh = ax.pcolormesh(
            da.lon, da.lat, data,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            shading="auto",
        )

    # --- 4. Add map features ---
    if landmask:
            land = cfeature.NaturalEarthFeature('physical', 'land', '110m', edgecolor='face')
            ax.add_feature(land, facecolor='lightgray')

    ax.coastlines(linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_global()  # ensures full globe is shown

    # Optional: gridlines
    gl = ax.gridlines(draw_labels=False, linewidth=0.3, color="gray", alpha=0.5)

    # # --- 5. Colorbar ---
    if cbar:
        cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.05, shrink=0.7)
        cbar.set_label(r"Spatial variance in pattern ($\sigma$)")

    # ax.set_title(f"Robinson Projection Centered at {central_lon}°E", fontsize=13)

    plt.tight_layout()
    # plt.savefig("robinson_plot.png", dpi=200, bbox_inches="tight")

    plt.title(da.name)

    # Make figure and axes backgrounds transparent
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    # Cartopy adds its own background patch for the map outline — clear that too
    # ax.background_patch.set_alpha(0)   # older cartopy versions
    # or, in newer cartopy versions:
    ax.spines['geo'].set_visible(False)  # optional: remove the border/outline too

    # Save as SVG with transparency
    plt.savefig(savepath, format="svg", transparent=True, bbox_inches="tight")
    # plt.show()

In [4]:
cdo = Cdo()
template_path           = '../amip/data/lowres_template.nc'



# cdo.remapcon(template_path,  input =  filepath_input_SST, output = filepath_output_SST, options = '-f nc')
# cdo.remapcon(template_path,  input =  filepath_input_ceres, output = filepath_output_ceres, options = '-f nc')
# cdo.remapcon(template_path,  input =  filepath_input_era5, output = filepath_output_era5, options = '-f nc')

In [5]:
# Remapping for these was done elsewhere
filepath_output_ceres   = '/scratch/leiff/data/obs/CERES/CERES_200303_202604_lowRes.nc'
filepath_output_era5    = '/scratch/leiff/data/obs/ERA5/tas_200303_202512_lowRes.nc'
ERA5    = xc.open_dataset(filepath_output_era5)
CERES   = xc.open_dataset(filepath_output_ceres)

In [13]:
N_full      = np.array(CERES['toa_net_all_mon'].sel(time=slice('2000-03','2025-12')).data)
print(N_full.shape)
N_amip      = np.array(CERES['toa_net_all_mon'].sel(time=slice('2000-03','2014-12')).data)
print(N_amip.shape)

tas_full    = np.array(ERA5['t2m'].sel(time=slice('2000-03','2025-12')).data)
print(tas_full.shape)
tas_amip    = np.array(ERA5['t2m'].sel(time=slice('2000-03','2014-12')).data)
print(tas_amip.shape)

(310, 64, 128)
(178, 64, 128)
(274, 64, 128)
(142, 64, 128)


In [7]:
# Ng_test  = CERES.spatial.average('toa_net_all_mon')['toa_net_all_mon']
CERES['weights']    = CERES['toa_net_all_mon'][0]*0+np.cos(np.deg2rad(CERES.lat))
CERES['Ng']         = np.sum(CERES['toa_net_all_mon']*CERES['weights'],axis=(1,2)) /np.sum(CERES['weights'])

# TG_test = ERA5.spatial.average('t2m')['t2m']
ERA5['weights'] = ERA5['t2m'][0]*0+np.cos(np.deg2rad(ERA5.lat))
ERA5['Tg']      = np.sum(ERA5['t2m']*ERA5['weights'],axis=(1,2)) /np.sum(ERA5['weights'])


Ng_full         = np.array(CERES['Ng'].sel(time=slice('2000-03','2025-12')).data)
print(Ng_full.shape)
Ng_amip      = np.array(CERES['Ng'].sel(time=slice('2000-03','2014-12')).data)
print(Ng_amip.shape)

Tg_full    = np.array(ERA5['Tg'].sel(time=slice('2000-03','2025-12')).data)
print(Tg_full.shape)
Tg_amip    = np.array(ERA5['Tg'].sel(time=slice('2000-03','2014-12')).data)
print(Tg_amip.shape)

(274,)
(142,)
(274,)
(142,)


In [8]:
tas_full_det    = detrend_monthly_fast(tas_full)
tas_amip_det    = detrend_monthly_fast(tas_amip)

Tg_full_det     = detrend_monthly_fast(Tg_full)
Tg_amip_det     = detrend_monthly_fast(Tg_amip)

N_full_det      = detrend_monthly_fast(N_full)
N_amip_det      = detrend_monthly_fast(N_amip)

Ng_full_det      = detrend_monthly_fast(Ng_full)
Ng_amip_det      = detrend_monthly_fast(Ng_amip)

In [9]:
tas_amip_reshaped=tas_amip_det.reshape(tas_amip_det.shape[0],-1)
tas_full_reshaped=tas_full_det.reshape(tas_full_det.shape[0],-1)
weights_1D  = np.array(ERA5['weights'].data).reshape(-1)
print(tas_amip_reshaped.shape)
print(tas_full_reshaped.shape)

(142, 8192)
(274, 8192)


In [10]:
ds_maps_obsVmodels  = xc.open_dataset(filepath_output_ceres).isel(time=0).drop_vars(('toa_sw_all_mon','toa_lw_all_mon',
'toa_net_all_mon','toa_sw_clr_c_mon','toa_lw_clr_c_mon','toa_net_clr_c_mon','solar_mon','cldarea_total_daynight_mon',
'cldpress_total_daynight_mon','cldtemp_total_daynight_mon'))
ds_maps_obsVmodels

<xarray.Dataset> Size: 37kB
Dimensions:               (lon: 128, bnds: 2, lat: 64)
Coordinates:
  * lon                   (lon) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * lat                   (lat) float64 512B -87.86 -85.1 -82.31 ... 85.1 87.86
    time                  object 8B 2000-03-15 00:00:00
Dimensions without coordinates: bnds
Data variables:
    lon_bnds              (lon, bnds) float64 2kB ...
    lat_bnds              (lat, bnds) float64 1kB ...
    cldtau_total_day_mon  (lat, lon) float32 33kB ...
Attributes:
    CDI:          Climate Data Interface version 2.5.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.4
    institution:  NASA/LaRC (Langley Research Center) Hampton, Va
    title:        CERES EBAF (Energy Balanced and Filled) TOA Fluxes. Monthly...
    comment:      Climatology from 07/2005 to 06/2015
    version:      Edition 4.2.1; Release Date November 25, 2024
    DOI:          10.5067/TERRA-AQUA-NOAA20/CERES/EBAF-TOA_L3B004.2.1
    history:      Tue Jul 28 10:45:49 2026: cdo -O -s -f nc -remapcon,./data/...
    CDO:          Climate Data Operators version 2.5.0 (https://mpimet.mpg.de...

In [12]:
ds_maps_obsVmodels['L_i_amip_std']   = (('lat','lon'), 
    MCA_dave(tas_amip_reshaped/np.std(tas_amip_reshaped,axis=0),
        Ng_amip_det,weights_1D).reshape(len(ds_maps_obsVmodels.lat),len(ds_maps_obsVmodels.lon)))
ds_maps_obsVmodels['L_i_full_std']   = (('lat','lon'), 
    MCA_dave(tas_full_reshaped/np.std(tas_full_reshaped,axis=0),
        Ng_full_det,weights_1D).reshape(len(ds_maps_obsVmodels.lat),len(ds_maps_obsVmodels.lon)))

In [ ]:
model_names = ['BCC-CSM2-MR','CAMS-CSM1-0','CESM2','CIESM','CNRM-CM6-1','CNRM-CM6-1-HR','CNRM-ESM2-1',
                'CanESM5','FGOALS-f3-L','FGOALS-g3','FIO-ESM-2-0',
                'IITM-ESM','IPSL-CM6A-LR','MIROC6','MRI-ESM2-0','TaiESM1']

n_models = len(model_names)

stop

for i in np.arange(n_models):

    # Apparently there is an extra year
    if model_names[i] == 'FGOALS-g3':
        filepath_tas    = '/rugenstein-archive/senne/data/AMIP/' + model_names[i] + '/month/tas_amip-hist_187001-201512.nc'
        filepath_N      = '/rugenstein-archive/senne/data/AMIP/' + model_names[i] + '/month/N_GM_amip-hist_187001-201512.nc'

        # tas_hist_ds   = xc.open_dataset(filepath_tas)
        # N_hist_ds     = xc.open_dataset(filepath_N)

        # tas_hist   = np.array(tas_hist_ds['tas'].data)[:,:-12]
        # N_hist     = np.array(N_hist_ds['N'].data)[:,:-12]     


    else:
        filepath_tas    = '/rugenstein-archive/senne/data/AMIP/' + model_names[i] + '/month/tas_amip-hist_187001-201412.nc'
        filepath_N      = '/rugenstein-archive/senne/data/AMIP/' + model_names[i] + '/month/N_GM_amip-hist_187001-201412.nc'

    tas_hist_ds   = xc.open_dataset(filepath_tas).sel(time=slice('2003-03','2014-12'))
    N_hist_ds     = xc.open_dataset(filepath_N).sel(time=slice('2003-03','2014-12'))

    tas_hist   = np.array(tas_hist_ds['tas'].data)
    N_hist     = np.array(N_hist_ds['N'].data)      

    # print(model_names[i])
    # print(tas_hist_ds.time[-1])
    # print(N_hist_ds.time[-1])
    # print(tas_hist_ds.time[0])
    # print(N_hist_ds.time[0])
    # print(tas_hist.shape)
    # print(N_hist.shape)

    # Now we get into the individual MCA map creation
    n_mon       = N_hist.shape[1]
    n_ens       = N_hist.shape[0]
    model_name  = model_names[i]

    print(n_mon,n_ens,model_name)

    MCA_m_std   = np.zeros((n_ens,tas_hist.shape[2]*tas_hist.shape[3]))
    for r in np.arange(n_ens):
        print('Model: '+model_name+f', ens {r}')
        tas_m       = tas_hist[r].reshape(n_mon,-1)
        N_m         = N_hist[r]
 
        tas_here    = detrend_monthly_fast(tas_m)
        N_here      = detrend_monthly_fast(N_m)

        MCA_m_std[r]   = MCA_dave(tas_here/np.std(tas_here,axis=0),N_here)

        # print('./data/maps/MCA_std_CERES_Period_'+model_name+f'_ens_{r}.npy')
        np.save('./data/maps/MCA_std_CERES_Period_'+model_name+f'_ens_{r}.npy',MCA_m_std[r])
        
        data = np.load(f'./data/maps/MCA_std_CERES_Period_{model_name}_ens_{r}.npy')
        ds_maps_obsVmodels[f'{model_name}_ens_{r}']  = (('lat','lon'),(data).reshape(len(ds_maps_obsVmodels.lat),
                                                                                    len(ds_maps_obsVmodels.lon)))
        svg_plot(ds_maps_obsVmodels[f'{model_name}_ens_{r}'],f'./plots/obs/L_i_amip_std_{model_name}_ens_{r}.svg',
            landmask=False,cmap = cmr.fusion_r,vmin=np.nan,vcenter=0,vmax=np.nan,cbar=True,norm_map=True)
    
    np.save(f'./data/maps/MCA_std_CERES_Period_{model_name}_ensMean.npy',MCA_m_std.mean(axis=0))
    
    data_mean = np.load(f'./data/maps/MCA_std_CERES_Period_{model_name}_ensMean.npy')

    ds_maps_obsVmodels[f'{model_name}_ensMean']  = (('lat','lon'),(data_mean).reshape(len(ds_maps_obsVmodels.lat),
                                                                                        len(ds_maps_obsVmodels.lon)))

    svg_plot(ds_maps_obsVmodels[f'{model_name}_ensMean'],f'./plots/obs/L_i_amip_std_{model_name}_ensMean.svg',
            landmask=False,cmap = cmr.fusion_r,vmin=np.nan,vcenter=0,vmax=np.nan,cbar=True,norm_map=True)
    # kill